# MLDS 422 Homework 6: Alumni lookup
I collect the alumni listed for 2013–2025, summarize their listed employers and roles, and save the table for a simple search tool.

Source: [Northwestern MLDS alumni directory](https://www.mccormick.northwestern.edu/machine-learning-data-science/people/alumni/). These are website listings, which may be outdated; they are not independently verified current jobs.

## 1. Set up
I use requests to download each page, BeautifulSoup to read its HTML, and pandas to organize the results. The data folder holds the saved table.

In [1]:
from pathlib import Path
from urllib.parse import urljoin
from datetime import date
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

DATA = Path('data')
DATA.mkdir(exist_ok=True)
BASE = 'https://www.mccormick.northwestern.edu/machine-learning-data-science/people/alumni/'

def read_page(url):
    """Download a page and stop clearly if the request fails."""
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return BeautifulSoup(response.content, 'html.parser')

print('Collection date:', date.today())

Collection date: 2026-11-16


## 2. Find the class pages
I get the links from the directory instead of guessing the URLs. The newer classes use a different URL layout. A dictionary keeps each year only once.

In [2]:
index = read_page(BASE)
class_pages = {}
for link in index.select('a[href]'):
    match = re.fullmatch(r'Class of (20\d{2})', link.get_text(strip=True))
    if match and 2013 <= int(match.group(1)) <= 2025:
        class_pages[int(match.group(1))] = urljoin(BASE, link['href'])

assert set(class_pages) == set(range(2013, 2026)), 'A required year is missing.'
print('Years found:', sorted(class_pages))

Years found: [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


All 13 class years were found. Using the actual links handles both old and new page layouts.

## 3. Read each alumni entry
Most job descriptions say “Role at Company,” so I split at the last “ at ”. I first tidy spaces, including nonbreaking spaces. For descriptions without that separator, a small role pattern handles titles such as “Data Scientist TikTok.” Standalone role descriptions stay in Role; company-only descriptions stay in Company. Blank details remain missing. The 2024 and 2025 cards only say “Graduate Student,” so I retain that label without inventing employers. I also save the original job text for checking.

In [3]:
def clean_text(text):
    """Turn extra whitespace into ordinary single spaces."""
    return ' '.join(text.split())


def split_job(text):
    """Separate the site's job wording without changing company names."""
    if not text:
        return pd.NA, pd.NA
    if ' at ' in text:
        return text.rsplit(' at ', 1)
    # Some cards omit "at" after these common role titles.
    match = re.fullmatch(r'(Data Scientist|Data Engineer) (.+)', text)
    if match:
        return match.group(1), match.group(2)
    role_words = r'student|scientist|engineer|manager|analyst|director|consultant|worker|self-employed|data science'
    if re.search(role_words, text, flags=re.I):
        return text, pd.NA
    return pd.NA, text


rows = []
audit_rows = []
for year, url in sorted(class_pages.items()):
    page = read_page(url)
    cards = page.select('.faculty-info')
    if not cards:
        raise ValueError(f'No alumni cards found for {year}.')
    for card in cards:
        name = clean_text(card.select_one('h3').get_text(' ', strip=True))
        details = card.select_one('.student-alumni-details')
        text = clean_text(details.get_text(' ', strip=True)) if details else ''
        role, company = split_job(text)
        if not text:
            label = card.select_one('em')
            if label and clean_text(label.text) == 'Graduate Student':
                role = 'Graduate Student'
        rows.append({'Name': name, 'Year': year, 'Role': role, 'Company': company})
        audit_rows.append({'Name': name, 'Year': year, 'Original job text': text, 'Source': url})
    print(f'{year}: {len(cards)} alumni')
    time.sleep(0.2)  # Leave a short pause between requests.

alumni = pd.DataFrame(rows)
audit = pd.DataFrame(audit_rows)
print('Total alumni:', len(alumni))
display(alumni.head())

2013: 30 alumni
2014: 31 alumni
2015: 31 alumni
2016: 38 alumni
2017: 39 alumni
2018: 41 alumni
2019: 46 alumni
2020: 47 alumni
2021: 41 alumni
2022: 42 alumni
2023: 49 alumni
2024: 52 alumni
2025: 57 alumni
Total alumni: 544


,Name,Year,Role,Company
0,Scott Albrecht,2013,"Senior Manager, Data Science",Grainger
1,Yoojong Bang,2013,Principal Data Scientist,Intent Media Inc.
2,Shawna Baskin,2013,Senior Manager - Tabletop Analytics,Wizards of the Coast
3,William Chiu,2013,"Vice President, Head of Statistical Modeling",First Republic Bank
4,David Cooperberg,2013,Senior Product Manager,Alteryx


The pages returned 544 alumni. Everyone is retained, including people without a listed job.

## 4. Check and save
I check that each name/year pair appears once and that all requested years are present. Missing jobs are allowed because the website does not list them for everyone. The pickle preserves the DataFrame for the lookup script; the CSV files make it easy to inspect on GitHub.

In [4]:
assert not alumni.duplicated(['Name', 'Year']).any()
assert set(alumni['Year']) == set(range(2013, 2026))
assert alumni['Name'].str.len().gt(0).all()
display(alumni.isna().sum().rename('Missing values').to_frame())
alumni.to_pickle(DATA / 'alumni.pkl')
alumni.to_csv(DATA / 'alumni.csv', index=False)
audit.to_csv(DATA / 'source_details.csv', index=False)
print('Saved data/alumni.pkl')

,Missing values
Name,0
Year,0
Role,41
Company,151


Saved data/alumni.pkl


The table is saved with exactly Name, Year, Role, and Company. Missing employment details are left unknown, so those records do not enter employer rankings.

## 5. Top five companies
I count alumni with a known company. Capitalization and extra spaces are ignored when grouping, but I do not merge subsidiaries or different company names. Ties are sorted alphabetically.

In [5]:
known = alumni.dropna(subset=['Company']).copy()
known['company_key'] = known['Company'].str.casefold()
company_counts = known.groupby('company_key').agg(
    Company=('Company', 'first'), Alumni=('Name', 'size')
).sort_values(['Alumni', 'Company'], ascending=[False, True])
top_five = company_counts.head(5).reset_index(drop=True)
display(top_five)
print('Top five listed employers:', ', '.join(
    f'{row.Company} ({row.Alumni})' for row in top_five.itertuples()
))

,Company,Alumni
0,TransUnion,15
1,Meta,13
2,Amazon,12
3,Facebook,12
4,Coupa Software,8


Top five listed employers: TransUnion (15), Meta (13), Amazon (12), Facebook (12), Coupa Software (8)


TransUnion has the most entries (15), followed by Meta (13), Amazon (12), Facebook (12), and Coupa Software (8). These counts use the names printed on the site: Facebook and Meta remain separate labels. Combining renamed companies would change this ranking, so this is a ranking of listed employer names, not corporate groups.

## 6. Insight one: Size of the listed classes
I compare how many alumni appear on each class page. This describes the directory, so it may not equal the number of people who actually graduated.

In [6]:
class_sizes = alumni.groupby('Year').size().rename('Listed alumni')
display(class_sizes.to_frame())
first, last = class_sizes.loc[2013], class_sizes.loc[2025]
print(f'The directory lists {first} alumni for 2013 and {last} for 2025.')
print(f'That is {last - first} more people, or {(last / first - 1) * 100:.1f}% more.')

,Listed alumni
Year,
2013,30
2014,31
2015,31
2016,38
2017,39
2018,41
2019,46
2020,47
2021,41


The directory lists 30 alumni for 2013 and 57 for 2025.
That is 27 more people, or 90.0% more.


The 2025 page lists 57 people compared with 30 in 2013, a 90% increase. The directory therefore contains a larger recent class, but this alone does not establish a trend in actual graduation totals.

## 7. Insight two: Senior roles
I look for the word “senior” in job titles and report the share among listed employment roles. I exclude the generic Graduate Student labels so they do not lower that percentage. This measures title wording, not everyone’s actual experience level.

In [7]:
employment_roles = alumni.loc[
    alumni['Role'].notna() & alumni['Role'].ne('Graduate Student')
].copy()
senior = employment_roles['Role'].str.contains(r'\bsenior\b', case=False, regex=True)
print(f'{senior.sum()} of {len(employment_roles)} listed employment roles contain "senior".')
print(f'That is {senior.mean() * 100:.1f}% of known employment roles.')
display(employment_roles.loc[senior].head())

87 of 394 listed employment roles contain "senior".
That is 22.1% of known employment roles.


,Name,Year,Role,Company
0,Scott Albrecht,2013,"Senior Manager, Data Science",Grainger
2,Shawna Baskin,2013,Senior Manager - Tabletop Analytics,Wizards of the Coast
4,David Cooperberg,2013,Senior Product Manager,Alteryx
5,Adam Evans,2013,Senior Data Analyst,Canva
31,Anirban Bhattacharyya,2014,Senior Data Scientist,Dropbox


87 of 394 known employment roles (22.1%) include “senior.” This suggests senior titles are common among listed jobs. It does not count other advanced titles such as director, and missing or outdated listings limit the conclusion.

## 8. Try the lookup class
The separate script reads the pickle and lets me search repeatedly. Here I check all four fields with mixed capitalization. Partial matches are literal, so punctuation is not treated as a regular expression.

In [8]:
from mlds_student_lookup import MLDSStudentLookup
lookup = MLDSStudentLookup(pd.read_pickle(DATA / 'alumni.pkl'))
for field, value in [('name', 'Scott'), ('YEAR', '2013'), ('Role', 'senior'), ('Company', 'Google')]:
    result = lookup.filter_by(field, value)
    print(f'{field} contains {value!r}: {len(result)} matches')
    display(result.head())

# An invalid field should produce a helpful message.
try:
    lookup.filter_by('salary', '100')
except ValueError as error:
    print('Invalid-field check:', error)
assert lookup.filter_by('Name', 'no-such-person-12345').empty
assert lookup.filter_by('Company', '[').empty
print('Lookup checks passed.')

name contains 'Scott': 1 matches


,Name,Year,Role,Company
0,Scott Albrecht,2013,"Senior Manager, Data Science",Grainger


YEAR contains '2013': 30 matches


,Name,Year,Role,Company
0,Scott Albrecht,2013,"Senior Manager, Data Science",Grainger
1,Yoojong Bang,2013,Principal Data Scientist,Intent Media Inc.
2,Shawna Baskin,2013,Senior Manager - Tabletop Analytics,Wizards of the Coast
3,William Chiu,2013,"Vice President, Head of Statistical Modeling",First Republic Bank
4,David Cooperberg,2013,Senior Product Manager,Alteryx


Role contains 'senior': 87 matches


,Name,Year,Role,Company
0,Scott Albrecht,2013,"Senior Manager, Data Science",Grainger
2,Shawna Baskin,2013,Senior Manager - Tabletop Analytics,Wizards of the Coast
4,David Cooperberg,2013,Senior Product Manager,Alteryx
5,Adam Evans,2013,Senior Data Analyst,Canva
31,Anirban Bhattacharyya,2014,Senior Data Scientist,Dropbox


Company contains 'Google': 5 matches


,Name,Year,Role,Company
14,Katharine (Matsumoto) Hardison,2013,"Business Intelligence Manager, People Analytics",Google
118,Lingxiao (Lynn) Ou,2016,Senior Data Scientist,Google
172,Johnny Chiu,2018,Data Scientist,Google
209,Ethel Shiqi Zhang,2018,Quantative User Experience Researcher - Data S...,Google
266,Megan Hazlett,2020,"Product Analyst, Data Science",Google


Invalid-field check: Choose Name, Year, Role, or Company.
Lookup checks passed.


The four searches work with case-insensitive, partial matching. The terminal script prints every matching row without shortening values and continues after an invalid field. Run `python mlds_student_lookup.py` and enter `exit` to stop.